### Part A – DataFrame Creation

In [0]:
%python
# Import required PySpark functions and types
from pyspark.sql.functions import col, round as spark_round, upper, lit, when, avg, max, min, sum as spark_sum, count, to_date, year, month, dayofmonth, current_date, current_timestamp, date_add, datediff, expr, lower, length, substring, concat_ws, replace, countDistinct, coalesce, dayofweek
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# 1. Create the Employee DataFrame
employee_data = [
    (101, "Alice", 25, "HR", 45000, "Chennai", "2022-01-15", 201),
    (102, "Bob", 30, "IT", 70000, "Bangalore", "2021-06-20", 202),
    (103, "Charlie", None, "IT", None, "Chennai", "2023-03-12", 202),
    (104, "David", 28, "Finance", 65000, "Mumbai", "2020-09-18", 203),
    (105, "Eva", 35, "HR", 80000, None, "2019-05-25", 201),
    (106, "Frank", 29, "Marketing", 55000, "Hyderabad", None, 204),
    (107, "Grace", 31, "Finance", None, "Pune", "2022-12-01", 203),
    (108, "Henry", 26, "IT", 60000, "Bangalore", "2024-01-10", None)
]
employee_columns = ["emp_id", "name", "age", "department", "salary", "city", "joining_date", "manager_id"]
employee_df = spark.createDataFrame(employee_data, employee_columns)
display(employee_data)

# 2. Create the Department DataFrame
department_data = [
    ("HR", "Chennai"), ("IT", "Bangalore"), ("Finance", "Mumbai"),
    ("Marketing", "Hyderabad"), ("Sales", "Delhi")
]
department_columns = ["dept_name", "location"]
department_df = spark.createDataFrame(department_data, department_columns)
display(department_data)

# 3. Create the Manager DataFrame
manager_data = [
    (201, "Robert"), (202, "Jennifer"), (203, "Michael"), (204, "Sophia")
]
manager_columns = ["manager_id", "manager_name"]
manager_df = spark.createDataFrame(manager_data, manager_columns)
display(manager_data)

_1,_2,_3,_4,_5,_6,_7,_8
101,Alice,25,HR,45000,Chennai,2022-01-15,201
102,Bob,30,IT,70000,Bangalore,2021-06-20,202
103,Charlie,null,IT,null,Chennai,2023-03-12,202
104,David,28,Finance,65000,Mumbai,2020-09-18,203
105,Eva,35,HR,80000,null,2019-05-25,201
106,Frank,29,Marketing,55000,Hyderabad,null,204
107,Grace,31,Finance,null,Pune,2022-12-01,203
108,Henry,26,IT,60000,Bangalore,2024-01-10,null


_1,_2
HR,Chennai
IT,Bangalore
Finance,Mumbai
Marketing,Hyderabad
Sales,Delhi


_1,_2
201,Robert
202,Jennifer
203,Michael
204,Sophia


### Part B – Select Operations

In [0]:
# 4. Select only emp_id, name, and salary
employee_df.select("emp_id", "name", "salary").show()

+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|   101|  Alice| 45000|
|   102|    Bob| 70000|
|   103|Charlie|  NULL|
|   104|  David| 65000|
|   105|    Eva| 80000|
|   106|  Frank| 55000|
|   107|  Grace|  NULL|
|   108|  Henry| 60000|
+------+-------+------+



In [0]:
# 5. Select all employees from the IT department
employee_df.filter(col("department") == "IT").show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 6. Select employee names and alias the column as Employee_Name
employee_df.select(col("name").alias("Employee_Name")).show()

+-------------+
|Employee_Name|
+-------------+
|        Alice|
|          Bob|
|      Charlie|
|        David|
|          Eva|
|        Frank|
|        Grace|
|        Henry|
+-------------+



In [0]:
# 7. Select salary after increasing it by 10%
employee_df.select("name", (col("salary") * 1.1).alias("increased_salary")).show()

+-------+-----------------+
|   name| increased_salary|
+-------+-----------------+
|  Alice|49500.00000000001|
|    Bob|          77000.0|
|Charlie|             NULL|
|  David|          71500.0|
|    Eva|          88000.0|
|  Frank|60500.00000000001|
|  Grace|             NULL|
|  Henry|          66000.0|
+-------+-----------------+



In [0]:
# 8. Select employees whose city is Chennai or Bangalore
employee_df.filter(col("city").isin("Chennai", "Bangalore")).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



### Part C – withColumn()

In [0]:
# 9. Create a new column called bonus equal to 20% of salary
employee_df.withColumn("bonus", col("salary") * 0.20).show()

+------+-------+----+----------+------+---------+------------+----------+-------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|  bonus|
+------+-------+----+----------+------+---------+------------+----------+-------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201| 9000.0|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|14000.0|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|   NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|13000.0|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|16000.0|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|11000.0|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|   NULL|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|12000.0|
+------+-------+----+----------+------+---------+------------+----------+-------+



In [0]:
# 10. Create a column called annual_salary
employee_df.withColumn("annual_salary", col("salary") * 12).show()

+------+-------+----+----------+------+---------+------------+----------+-------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|annual_salary|
+------+-------+----+----------+------+---------+------------+----------+-------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       540000|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       840000|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|         NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|       780000|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       960000|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|       660000|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|         NULL|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|       720000|
+------+-------+----+----------+

In [0]:
# 11. Create experience_bonus
employee_df.withColumn("experience_bonus",
    when(col("salary") > 70000, 10000)
    .when((col("salary") >= 50000) & (col("salary") <= 70000), 5000)
    .otherwise(2000)
).show()

+------+-------+----+----------+------+---------+------------+----------+----------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|experience_bonus|
+------+-------+----+----------+------+---------+------------+----------+----------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|            2000|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|            5000|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|            2000|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|            5000|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|           10000|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|            5000|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|            2000|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|            5000|

In [0]:
# 12. Convert employee names to uppercase
employee_df.withColumn("name", upper(col("name"))).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  ALICE|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    BOB|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|CHARLIE|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  DAVID|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    EVA|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  FRANK|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  GRACE|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  HENRY|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 13. Add a column called country with value "India"
employee_df.withColumn("country", lit("India")).show()

+------+-------+----+----------+------+---------+------------+----------+-------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|country|
+------+-------+----+----------+------+---------+------------+----------+-------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|  India|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|  India|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|  India|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  India|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|  India|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|  India|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  India|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|  India|
+------+-------+----+----------+------+---------+------------+----------+-------+



### Part D – withColumnRenamed()

In [0]:
# 14. Rename department to dept
employee_df.withColumnRenamed("department", "dept").show()

+------+-------+----+---------+------+---------+------------+----------+
|emp_id|   name| age|     dept|salary|     city|joining_date|manager_id|
+------+-------+----+---------+------+---------+------------+----------+
|   101|  Alice|  25|       HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|       IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|       IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|  Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|       HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29|Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|  Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|       IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+---------+------+---------+------------+----------+



In [0]:
# 15. Rename salary to monthly_salary
employee_df.withColumnRenamed("salary", "monthly_salary").show()

+------+-------+----+----------+--------------+---------+------------+----------+
|emp_id|   name| age|department|monthly_salary|     city|joining_date|manager_id|
+------+-------+----+----------+--------------+---------+------------+----------+
|   101|  Alice|  25|        HR|         45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT|         70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|          NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance|         65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR|         80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing|         55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|          NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT|         60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+--------------+---------+------------+----------+



In [0]:
# 16. Rename multiple columns one by one
employee_df.withColumnRenamed("name", "full_name") \
           .withColumnRenamed("age", "employee_age").show()

+------+---------+------------+----------+------+---------+------------+----------+
|emp_id|full_name|employee_age|department|salary|     city|joining_date|manager_id|
+------+---------+------------+----------+------+---------+------------+----------+
|   101|    Alice|          25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|      Bob|          30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|  Charlie|        NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|    David|          28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|      Eva|          35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|    Frank|          29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|    Grace|          31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|    Henry|          26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+---------+------------+----------+------+---------+------------+----

### Part E – Filter

In [0]:
# 17. Find employees earning more than 60000
employee_df.filter(col("salary") > 60000).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|  Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 18. Find employees older than 30
employee_df.filter(col("age") > 30).show()

+------+-----+---+----------+------+----+------------+----------+
|emp_id| name|age|department|salary|city|joining_date|manager_id|
+------+-----+---+----------+------+----+------------+----------+
|   105|  Eva| 35|        HR| 80000|NULL|  2019-05-25|       201|
|   107|Grace| 31|   Finance|  NULL|Pune|  2022-12-01|       203|
+------+-----+---+----------+------+----+------------+----------+



In [0]:
# 19. Find HR employees in Chennai
employee_df.filter((col("department") == "HR") & (col("city") == "Chennai")).show()

+------+-----+---+----------+------+-------+------------+----------+
|emp_id| name|age|department|salary|   city|joining_date|manager_id|
+------+-----+---+----------+------+-------+------------+----------+
|   101|Alice| 25|        HR| 45000|Chennai|  2022-01-15|       201|
+------+-----+---+----------+------+-------+------------+----------+



In [0]:
# 20. Find employees with NULL salary
employee_df.filter(col("salary").isNull()).show()

+------+-------+----+----------+------+-------+------------+----------+
|emp_id|   name| age|department|salary|   city|joining_date|manager_id|
+------+-------+----+----------+------+-------+------------+----------+
|   103|Charlie|NULL|        IT|  NULL|Chennai|  2023-03-12|       202|
|   107|  Grace|  31|   Finance|  NULL|   Pune|  2022-12-01|       203|
+------+-------+----+----------+------+-------+------------+----------+



In [0]:
# 21. Find employees whose salary is between 50000 and 70000
employee_df.filter(col("salary").between(50000, 70000)).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   106|Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 22. Find employees whose names start with "A"
employee_df.filter(col("name").startswith("A")).show()

+------+-----+---+----------+------+-------+------------+----------+
|emp_id| name|age|department|salary|   city|joining_date|manager_id|
+------+-----+---+----------+------+-------+------------+----------+
|   101|Alice| 25|        HR| 45000|Chennai|  2022-01-15|       201|
+------+-----+---+----------+------+-------+------------+----------+



In [0]:
# 23. Find employees whose city is not NULL
employee_df.filter(col("city").isNotNull()).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



### Part F – Sort

In [0]:
# 24. Sort employees by salary ascending
employee_df.orderBy("salary").show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 25. Sort employees by salary descending
employee_df.orderBy(col("salary").desc()).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 26. Sort employees by department and salary
employee_df.orderBy("department", "salary").show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 27. Sort employees by age descending
employee_df.orderBy(col("age").desc()).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 28. Display top 3 highest-paid employees
employee_df.orderBy(col("salary").desc()).limit(3).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   105|  Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
+------+-----+---+----------+------+---------+------------+----------+



### Part G – FillNA and DropNA

In [0]:
# 29. Replace NULL salary with 30000
employee_df.fillna({"salary": 30000}).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT| 30000|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance| 30000|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 30. Replace NULL city with "Unknown"
employee_df.fillna({"city": "Unknown"}).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|  Unknown|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 31. Replace NULL age with average age
avg_age = employee_df.select(avg("age")).collect()[0][0]
employee_df.fillna({"age": avg_age}).show()

+------+-------+---+----------+------+---------+------------+----------+
|emp_id|   name|age|department|salary|     city|joining_date|manager_id|
+------+-------+---+----------+------+---------+------------+----------+
|   101|  Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie| 29|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace| 31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+---+----------+------+---------+------------+----------+



In [0]:
# 32. Replace multiple NULL columns in one statement
employee_df.fillna({"salary": 0, "city": "N/A", "age": 0}).show()

+------+-------+---+----------+------+---------+------------+----------+
|emp_id|   name|age|department|salary|     city|joining_date|manager_id|
+------+-------+---+----------+------+---------+------------+----------+
|   101|  Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|  0|        IT|     0|  Chennai|  2023-03-12|       202|
|   104|  David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva| 35|        HR| 80000|      N/A|  2019-05-25|       201|
|   106|  Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace| 31|   Finance|     0|     Pune|  2022-12-01|       203|
|   108|  Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+---+----------+------+---------+------------+----------+



In [0]:
# 33. Drop rows containing any NULL values
employee_df.dropna(how="any").show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   101|Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 34. Drop rows where salary is NULL
employee_df.dropna(subset=["salary"]).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   101|Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|  Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 35. Drop rows only if all values are NULL
employee_df.dropna(how="all").show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



### Part H – Date Functions

In [0]:
# Prepare DF for date operations (saving to a new variable for reuse)
emp_dates_df = employee_df.withColumn("joining_date", to_date(col("joining_date"), "yyyy-MM-dd"))

In [0]:
# 36. Convert joining_date to DateType
emp_dates_df.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- manager_id: long (nullable = true)



In [0]:
# 37. Extract joining year
emp_dates_df.withColumn("joining_year", year(col("joining_date"))).show()

+------+-------+----+----------+------+---------+------------+----------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|joining_year|
+------+-------+----+----------+------+---------+------------+----------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|        2022|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|        2021|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|        2023|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|        2020|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|        2019|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|        NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|        2022|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|        2024|
+------+-------+----+----------+------+----

In [0]:
# 38. Extract joining month
emp_dates_df.withColumn("joining_month", month(col("joining_date"))).show()

+------+-------+----+----------+------+---------+------------+----------+-------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|joining_month|
+------+-------+----+----------+------+---------+------------+----------+-------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|            1|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|            6|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|            3|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|            9|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|            5|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|         NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|           12|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|            1|
+------+-------+----+----------+

In [0]:
# 39. Extract joining day
emp_dates_df.withColumn("joining_day", dayofmonth(col("joining_date"))).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|joining_day|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|         15|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|         20|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|         12|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|         18|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|         25|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|       NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|          1|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|         10|
+------+-------+----+----------+------+---------+-----

In [0]:
# 40. Find employees who joined after 2022
emp_dates_df.filter(year(col("joining_date")) > 2022).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 41. Calculate years worked
emp_dates_df.withColumn("years_worked", spark_round(datediff(current_date(), col("joining_date")) / 365, 1)).show()

+------+-------+----+----------+------+---------+------------+----------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|years_worked|
+------+-------+----+----------+------+---------+------------+----------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|         4.5|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|         5.1|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|         3.4|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|         5.8|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|         7.2|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|        NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|         3.6|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|         2.5|
+------+-------+----+----------+------+----

In [0]:
# 42. Find employees who joined this year
emp_dates_df.filter(year(col("joining_date")) == year(current_date())).show()

+------+----+---+----------+------+----+------------+----------+
|emp_id|name|age|department|salary|city|joining_date|manager_id|
+------+----+---+----------+------+----+------------+----------+
+------+----+---+----------+------+----+------------+----------+



In [0]:
# 43. Display current date
emp_dates_df.withColumn("today", current_date()).show()

+------+-------+----+----------+------+---------+------------+----------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|     today|
+------+-------+----+----------+------+---------+------------+----------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|2026-07-24|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|2026-07-24|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|2026-07-24|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|2026-07-24|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|2026-07-24|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|2026-07-24|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|2026-07-24|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|2026-07-24|
+------+-------+----+----------+------+---------+------------+---

In [0]:
# 44. Display current timestamp
emp_dates_df.withColumn("now", current_timestamp()).show()

+------+-------+----+----------+------+---------+------------+----------+--------------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|                 now|
+------+-------+----+----------+------+---------+------------+----------+--------------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|2026-07-24 16:12:...|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|2026-07-24 16:12:...|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|2026-07-24 16:12:...|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|2026-07-24 16:12:...|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|2026-07-24 16:12:...|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|2026-07-24 16:12:...|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|2026-07-24 16:12:...|
|   108|  Henry|  26|        IT| 60000|Bangalore| 

In [0]:
# 45. Add 30 days to joining date
emp_dates_df.withColumn("plus_30_days", date_add(col("joining_date"), 30)).show()

+------+-------+----+----------+------+---------+------------+----------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|plus_30_days|
+------+-------+----+----------+------+---------+------------+----------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|  2022-02-14|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|  2021-07-20|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|  2023-04-11|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  2020-10-18|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|  2019-06-24|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|        NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  2022-12-31|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|  2024-02-09|
+------+-------+----+----------+------+----

In [0]:
# 46. Find the difference between joining date and today's date
emp_dates_df.withColumn("days_worked", datediff(current_date(), col("joining_date"))).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|days_worked|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       1651|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       1860|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       1230|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|       2135|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       2617|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|       NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|       1331|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|        926|
+------+-------+----+----------+------+---------+-----

### Part I – Joins

In [0]:
%python
# 47. Perform an inner join between Employee and Department
employee_df.join(department_df, employee_df.department == department_df.dept_name, "inner").show()

+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       IT|Bangalore|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       HR|  Chennai|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|Marketing|Hyderabad|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  Finance|   Mumbai|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-1

In [0]:
# 48. Perform a left join
employee_df.join(department_df, employee_df.department == department_df.dept_name, "left").show()

+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       IT|Bangalore|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       HR|  Chennai|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|Marketing|Hyderabad|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  Finance|   Mumbai|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-1

In [0]:
# 49. Perform a right join
employee_df.join(department_df, employee_df.department == department_df.dept_name, "right").show()

+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       HR|  Chennai|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|       IT|Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       IT|Bangalore|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  Finance|   Mumbai|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NUL

In [0]:
# 50. Perform a full outer join
employee_df.join(department_df, employee_df.department == department_df.dept_name, "outer").show()

+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       IT|Bangalore|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       HR|  Chennai|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|Marketing|Hyderabad|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  Finance|   Mumbai|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-1

In [0]:
# 51. Join Employee with Manager DataFrame
employee_df.join(manager_df, "manager_id", "left").show()

+----------+------+-------+----+----------+------+---------+------------+------------+
|manager_id|emp_id|   name| age|department|salary|     city|joining_date|manager_name|
+----------+------+-------+----+----------+------+---------+------------+------------+
|       201|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|      Robert|
|       202|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|    Jennifer|
|       202|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|    Jennifer|
|       203|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|     Michael|
|       201|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|      Robert|
|       204|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|      Sophia|
|       203|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|     Michael|
|      NULL|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|        NULL|
+----------+------+-------+----+----------+

In [0]:
# 52. Display employee name along with manager name
employee_df.join(manager_df, "manager_id", "left").select("name", "manager_name").show()

+-------+------------+
|   name|manager_name|
+-------+------------+
|  Alice|      Robert|
|    Bob|    Jennifer|
|Charlie|    Jennifer|
|  David|     Michael|
|    Eva|      Robert|
|  Frank|      Sophia|
|  Grace|     Michael|
|  Henry|        NULL|
+-------+------------+



In [0]:
# 53. Find employees whose department location matches their city
employee_df.join(department_df, (employee_df.department == department_df.dept_name) & (employee_df.city == department_df.location), "inner").show()

+------+-----+---+----------+------+---------+------------+----------+---------+---------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-----+---+----------+------+---------+------------+----------+---------+---------+
|   101|Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   106|Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|Marketing|Hyderabad|
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|       IT|Bangalore|
+------+-----+---+----------+------+---------+------------+----------+---------+---------+



In [0]:

# 54. Find departments having no employees
department_df.join(employee_df, department_df.dept_name == employee_df.department, "left_anti").show()

+---------+--------+
|dept_name|location|
+---------+--------+
|    Sales|   Delhi|
+---------+--------+



### Part J – Union

In [0]:
# Create another Employee DataFrame for Union
new_employee_data = [
    (109, "Irene", 27, "HR", 52000, "Chennai", "2024-03-11", 201),
    (110, "Jack", 34, "Sales", 72000, "Delhi", "2021-11-20", 205)
]
new_employee_df = spark.createDataFrame(new_employee_data, employee_columns)
display(new_employee_data)

_1,_2,_3,_4,_5,_6,_7,_8
109,Irene,27,HR,52000,Chennai,2024-03-11,201
110,Jack,34,Sales,72000,Delhi,2021-11-20,205


In [0]:
# 55. Union the two DataFrames (saving to variable for next questions)
union_df = employee_df.union(new_employee_df)
union_df.show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   109|  Irene|  27|        HR| 52000|  Chennai|  2024-03-11|       201|
|   110|   Jack|  34|     Sales| 72000|    Delhi|  2021-11-20|       205|
+------+-------+----+----------+------

In [0]:
# 56. Count total employees after union
print(f"Total employees: {union_df.count()}")

Total employees: 10


In [0]:
# 57. Remove duplicate employees after union
union_df.distinct().show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   109|  Irene|  27|        HR| 52000|  Chennai|  2024-03-11|       201|
|   110|   Jack|  34|     Sales| 72000|    Delhi|  2021-11-20|       205|
+------+-------+----+----------+------

In [0]:
# 58. Union by column names
employee_df.unionByName(new_employee_df).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   109|  Irene|  27|        HR| 52000|  Chennai|  2024-03-11|       201|
|   110|   Jack|  34|     Sales| 72000|    Delhi|  2021-11-20|       205|
+------+-------+----+----------+------

### Part K – Other Transformations

In [0]:
# 59. Display distinct departments
employee_df.select("department").distinct().show()

+----------+
|department|
+----------+
|        HR|
|        IT|
|   Finance|
| Marketing|
+----------+



In [0]:
# 60. Count employees in each department
employee_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|        IT|    3|
|   Finance|    2|
| Marketing|    1|
+----------+-----+



In [0]:
# 61. Calculate average salary by department
employee_df.groupBy("department").agg(avg("salary").alias("avg_salary")).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|        HR|   62500.0|
|        IT|   65000.0|
|   Finance|   65000.0|
| Marketing|   55000.0|
+----------+----------+



In [0]:
# 62. Find maximum salary by department
employee_df.groupBy("department").agg(max("salary").alias("max_salary")).show()

+----------+----------+
|department|max_salary|
+----------+----------+
|        HR|     80000|
|        IT|     70000|
|   Finance|     65000|
| Marketing|     55000|
+----------+----------+



In [0]:
# 63. Find minimum salary by department
employee_df.groupBy("department").agg(min("salary").alias("min_salary")).show()

+----------+----------+
|department|min_salary|
+----------+----------+
|        HR|     45000|
|        IT|     60000|
|   Finance|     65000|
| Marketing|     55000|
+----------+----------+



In [0]:
# 64. Find total salary paid in each department
employee_df.groupBy("department").agg(spark_sum("salary").alias("total_salary")).show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        HR|      125000|
|        IT|      130000|
|   Finance|       65000|
| Marketing|       55000|
+----------+------------+



In [0]:
# 65. Count employees in each city
employee_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|    2|
|Bangalore|    2|
|   Mumbai|    1|
|     NULL|    1|
|Hyderabad|    1|
|     Pune|    1|
+---------+-----+



In [0]:
# 66. Display unique cities
employee_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|  Chennai|
|Bangalore|
|   Mumbai|
|     NULL|
|Hyderabad|
|     Pune|
+---------+



In [0]:
# 67. Remove duplicate records
employee_df.dropDuplicates().show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 68. Drop the manager_id column
employee_df.drop("manager_id").show()

+------+-------+----+----------+------+---------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|
+------+-------+----+----------+------+---------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|
+------+-------+----+----------+------+---------+------------+



### Part L – String Functions

In [0]:
# 69. Convert employee names to lowercase
employee_df.withColumn("name", lower(col("name"))).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  david|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 70. Find the length of each employee name
employee_df.withColumn("name_length", length(col("name"))).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|name_length|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|          5|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|          3|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|          7|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|          5|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|          3|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|          5|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|          5|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|          5|
+------+-------+----+----------+------+---------+-----

In [0]:
# 71. Extract the first three letters of employee names
employee_df.withColumn("name_prefix", substring(col("name"), 1, 3)).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|name_prefix|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|        Ali|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|        Bob|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|        Cha|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|        Dav|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|        Eva|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|        Fra|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|        Gra|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|        Hen|
+------+-------+----+----------+------+---------+-----

In [0]:
# 72. Concatenate employee name and city
employee_df.withColumn("name_city", concat_ws("_", col("name"), col("city"))).show()

+------+-------+----+----------+------+---------+------------+----------+---------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|      name_city|
+------+-------+----+----------+------+---------+------------+----------+---------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|  Alice_Chennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|  Bob_Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|Charlie_Chennai|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|   David_Mumbai|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|            Eva|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|Frank_Hyderabad|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|     Grace_Pune|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|Henry_Bangalore|
+------+--

In [0]:
# 73. Replace all occurrences of "a" with "@" in names
employee_df.withColumn("name_mod", replace(col("name"), lit("a"), lit("@"))).show()

+------+-------+----+----------+------+---------+------------+----------+--------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|name_mod|
+------+-------+----+----------+------+---------+------------+----------+--------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|   Alice|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|     Bob|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202| Ch@rlie|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|   D@vid|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|     Ev@|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|   Fr@nk|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|   Gr@ce|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|   Henry|
+------+-------+----+----------+------+---------+------------+----------+--------+



### Part M – Aggregations

In [0]:
# 74. Find total salary
employee_df.select(spark_sum("salary").alias("total_payroll")).show()

+-------------+
|total_payroll|
+-------------+
|       375000|
+-------------+



In [0]:
# 75. Find average salary
employee_df.select(avg("salary").alias("average_payroll")).show()

+---------------+
|average_payroll|
+---------------+
|        62500.0|
+---------------+



In [0]:
# 76. Find highest-paid employee
employee_df.orderBy(col("salary").desc_nulls_last()).limit(1).show()

+------+----+---+----------+------+----+------------+----------+
|emp_id|name|age|department|salary|city|joining_date|manager_id|
+------+----+---+----------+------+----+------------+----------+
|   105| Eva| 35|        HR| 80000|NULL|  2019-05-25|       201|
+------+----+---+----------+------+----+------------+----------+



In [0]:
# 77. Find lowest-paid employee
employee_df.orderBy(col("salary").asc_nulls_last()).limit(1).show()

+------+-----+---+----------+------+-------+------------+----------+
|emp_id| name|age|department|salary|   city|joining_date|manager_id|
+------+-----+---+----------+------+-------+------------+----------+
|   101|Alice| 25|        HR| 45000|Chennai|  2022-01-15|       201|
+------+-----+---+----------+------+-------+------------+----------+



In [0]:
# 78. Count employees with salary greater than 50000
count_50k = employee_df.filter(col("salary") > 50000).count()
print(f"Employees earning > 50k: {count_50k}")

Employees earning > 50k: 5


### Part N – Advanced Questions

In [0]:
# 79. Display employees whose salary is above the department average
window_dept = Window.partitionBy("department")
employee_df.withColumn("dept_avg", avg("salary").over(window_dept)) \
           .filter(col("salary") > col("dept_avg")).show()

+------+----+---+----------+------+---------+------------+----------+--------+
|emp_id|name|age|department|salary|     city|joining_date|manager_id|dept_avg|
+------+----+---+----------+------+---------+------------+----------+--------+
|   102| Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202| 65000.0|
|   105| Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201| 62500.0|
+------+----+---+----------+------+---------+------------+----------+--------+



In [0]:
# 80. Find the second highest salary
window_rn = Window.orderBy(col("salary").desc())
employee_df.withColumn("rn", F.dense_rank().over(window_rn)) \
           .filter(col("rn") == 2).drop("rn").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----+---+----------+------+---------+------------+----------+
|emp_id|name|age|department|salary|     city|joining_date|manager_id|
+------+----+---+----------+------+---------+------------+----------+
|   102| Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
+------+----+---+----------+------+---------+------------+----------+



In [0]:
# 81. Find duplicate cities
employee_df.groupBy("city").count().filter(col("count") > 1).show()

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|    2|
|Bangalore|    2|
+---------+-----+



In [0]:
# 82. Display employees who joined in the last two years
emp_dates_df.filter(year(col("joining_date")) >= (year(current_date()) - 2)).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 83. Replace NULL manager IDs with 999
employee_df.fillna({"manager_id": 999}).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|       999|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 84. Create a salary grade
employee_df.withColumn("salary_grade",
    when(col("salary") >= 80000, "A")
    .when(col("salary") >= 60000, "B")
    .when(col("salary") >= 40000, "C")
    .otherwise("D")
).show()
display(employee_df)

+------+-------+----+----------+------+---------+------------+----------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|salary_grade|
+------+-------+----+----------+------+---------+------------+----------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|           C|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|           B|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|           D|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|           B|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|           A|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|           C|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|           D|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|           B|
+------+-------+----+----------+------+----

emp_id,name,age,department,salary,city,joining_date,manager_id
101,Alice,25,HR,45000,Chennai,2022-01-15,201
102,Bob,30,IT,70000,Bangalore,2021-06-20,202
103,Charlie,null,IT,null,Chennai,2023-03-12,202
104,David,28,Finance,65000,Mumbai,2020-09-18,203
105,Eva,35,HR,80000,null,2019-05-25,201
106,Frank,29,Marketing,55000,Hyderabad,null,204
107,Grace,31,Finance,null,Pune,2022-12-01,203
108,Henry,26,IT,60000,Bangalore,2024-01-10,null


In [0]:
# 85. Create an employee ID string like EMP101
employee_df.withColumn("emp_string", F.concat(lit("EMP"), col("emp_id"))).show()

+------+-------+----+----------+------+---------+------------+----------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|emp_string|
+------+-------+----+----------+------+---------+------------+----------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|    EMP101|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|    EMP102|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|    EMP103|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|    EMP104|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|    EMP105|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|    EMP106|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|    EMP107|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|    EMP108|
+------+-------+----+----------+------+---------+------------+---

In [0]:
# 86. Find employees whose names end with "e"
employee_df.filter(col("name").endswith("e")).show()

+------+-------+----+----------+------+-------+------------+----------+
|emp_id|   name| age|department|salary|   city|joining_date|manager_id|
+------+-------+----+----------+------+-------+------------+----------+
|   101|  Alice|  25|        HR| 45000|Chennai|  2022-01-15|       201|
|   103|Charlie|NULL|        IT|  NULL|Chennai|  2023-03-12|       202|
|   107|  Grace|  31|   Finance|  NULL|   Pune|  2022-12-01|       203|
+------+-------+----+----------+------+-------+------------+----------+



In [0]:
# 87. Display employees sorted by joining date
employee_df.orderBy(col("joining_date").asc_nulls_last()).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 88. Find employees who have worked for more than three years
emp_dates_df.withColumn("years_worked", datediff(current_date(), col("joining_date")) / 365) \
            .filter(col("years_worked") > 3).show()

+------+-------+----+----------+------+---------+------------+----------+------------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|      years_worked|
+------+-------+----+----------+------+---------+------------+----------+------------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201| 4.523287671232877|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202| 5.095890410958904|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|3.3698630136986303|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|5.8493150684931505|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|  7.16986301369863|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|3.6465753424657534|
+------+-------+----+----------+------+---------+------------+----------+------------------+



### Challenge Questions

In [0]:
# 89. Rank employees by salary within each department
employee_df.withColumn("salary_rank", F.rank().over(Window.partitionBy("department").orderBy(col("salary").desc()))).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|salary_rank|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|          1|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|          2|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|          1|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|          2|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|          1|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|          2|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|          3|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|          1|
+------+-------+----+----------+------+---------+-----

In [0]:
# 90. Find the top 2 highest-paid employees in every department
ranked_df = employee_df.withColumn("salary_rank", F.rank().over(Window.partitionBy("department").orderBy(col("salary").desc())))
ranked_df.filter(col("salary_rank") <= 2).show()

+------+-----+---+----------+------+---------+------------+----------+-----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|salary_rank|
+------+-----+---+----------+------+---------+------------+----------+-----------+
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|          1|
|   107|Grace| 31|   Finance|  NULL|     Pune|  2022-12-01|       203|          2|
|   105|  Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|          1|
|   101|Alice| 25|        HR| 45000|  Chennai|  2022-01-15|       201|          2|
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|          1|
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|          2|
|   106|Frank| 29| Marketing| 55000|Hyderabad|        NULL|       204|          1|
+------+-----+---+----------+------+---------+------------+----------+-----------+



In [0]:
# 91. Find departments with average salary greater than 60000
employee_df.groupBy("department").agg(avg("salary").alias("avg_sal")).filter(col("avg_sal") > 60000).show()

+----------+-------+
|department|avg_sal|
+----------+-------+
|        HR|62500.0|
|        IT|65000.0|
|   Finance|65000.0|
+----------+-------+



In [0]:
# 92. Find employees who do not have a manager
employee_df.filter(col("manager_id").isNull()).show()

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   108|Henry| 26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-----+---+----------+------+---------+------------+----------+



In [0]:
# 93. Display employees with manager names and department locations in a single DataFrame
employee_df.join(manager_df, "manager_id", "left") \
           .join(department_df, employee_df.department == department_df.dept_name, "left") \
           .select("name", "manager_name", "location").show()

+-------+------------+---------+
|   name|manager_name| location|
+-------+------------+---------+
|  Alice|      Robert|  Chennai|
|    Bob|    Jennifer|Bangalore|
|Charlie|    Jennifer|Bangalore|
|  David|     Michael|   Mumbai|
|    Eva|      Robert|  Chennai|
|  Frank|      Sophia|Hyderabad|
|  Grace|     Michael|   Mumbai|
|  Henry|        NULL|Bangalore|
+-------+------------+---------+



In [0]:
# 94. Identify employees with missing information (NULL values)
cond = " OR ".join([f"{c} IS NULL" for c in employee_df.columns])
employee_df.filter(expr(cond)).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 95. Replace missing salaries with the department average salary
window_dept = Window.partitionBy("department")
employee_df.withColumn("dept_avg", avg("salary").over(window_dept)) \
           .withColumn("salary", coalesce(col("salary"), col("dept_avg"))).drop("dept_avg").show()

+------+-------+----+----------+-------+---------+------------+----------+
|emp_id|   name| age|department| salary|     city|joining_date|manager_id|
+------+-------+----+----------+-------+---------+------------+----------+
|   104|  David|  28|   Finance|65000.0|   Mumbai|  2020-09-18|       203|
|   107|  Grace|  31|   Finance|65000.0|     Pune|  2022-12-01|       203|
|   101|  Alice|  25|        HR|45000.0|  Chennai|  2022-01-15|       201|
|   105|    Eva|  35|        HR|80000.0|     NULL|  2019-05-25|       201|
|   102|    Bob|  30|        IT|70000.0|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|65000.0|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT|60000.0|Bangalore|  2024-01-10|      NULL|
|   106|  Frank|  29| Marketing|55000.0|Hyderabad|        NULL|       204|
+------+-------+----+----------+-------+---------+------------+----------+



In [0]:
# 96. Calculate monthly tax as 10% of salary
employee_df.withColumn("monthly_tax", col("salary") * 0.10).show()

+------+-------+----+----------+------+---------+------------+----------+-----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|monthly_tax|
+------+-------+----+----------+------+---------+------------+----------+-----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|     4500.0|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|     7000.0|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|     6500.0|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|     8000.0|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|     5500.0|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|       NULL|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|     6000.0|
+------+-------+----+----------+------+---------+-----

In [0]:
# 97. Find employees who joined on weekends (1=Sunday, 7=Saturday in PySpark dayofweek)
emp_dates_df.filter(dayofweek(col("joining_date")).isin(1, 7)).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
+------+-------+----+----------+------+---------+------------+----------+



In [0]:
# 98. Create a final report
emp_dates_df.join(manager_df, "manager_id", "left") \
    .join(department_df, emp_dates_df.department == department_df.dept_name, "left") \
    .withColumn("Bonus", coalesce(col("salary"), lit(0)) * 0.20) \
    .withColumn("Annual_Salary", coalesce(col("salary"), lit(0)) * 12) \
    .withColumn("Years_Worked", spark_round(datediff(current_date(), col("joining_date")) / 365, 1)) \
    .withColumn("Salary_Grade", 
        when(col("salary") >= 80000, "A")
        .when(col("salary") >= 60000, "B")
        .when(col("salary") >= 40000, "C")
        .otherwise("D")
    ).select(
        col("name").alias("Employee_Name"),
        col("department").alias("Department"),
        col("manager_name").alias("Manager_Name"),
        col("location").alias("Department_Location"),
        col("salary").alias("Salary"),
        col("Bonus"),
        col("Annual_Salary"),
        col("Years_Worked"),
        col("Salary_Grade")
    ).show()
display(emp_dates_df)

+-------------+----------+------------+-------------------+------+-------+-------------+------------+------------+
|Employee_Name|Department|Manager_Name|Department_Location|Salary|  Bonus|Annual_Salary|Years_Worked|Salary_Grade|
+-------------+----------+------------+-------------------+------+-------+-------------+------------+------------+
|        Alice|        HR|      Robert|            Chennai| 45000| 9000.0|       540000|         4.5|           C|
|          Bob|        IT|    Jennifer|          Bangalore| 70000|14000.0|       840000|         5.1|           B|
|      Charlie|        IT|    Jennifer|          Bangalore|  NULL|    0.0|            0|         3.4|           D|
|        David|   Finance|     Michael|             Mumbai| 65000|13000.0|       780000|         5.8|           B|
|          Eva|        HR|      Robert|            Chennai| 80000|16000.0|       960000|         7.2|           A|
|        Frank| Marketing|      Sophia|          Hyderabad| 55000|11000.0|      

emp_id,name,age,department,salary,city,joining_date,manager_id
101,Alice,25,HR,45000,Chennai,2022-01-15,201
102,Bob,30,IT,70000,Bangalore,2021-06-20,202
103,Charlie,null,IT,null,Chennai,2023-03-12,202
104,David,28,Finance,65000,Mumbai,2020-09-18,203
105,Eva,35,HR,80000,null,2019-05-25,201
106,Frank,29,Marketing,55000,Hyderabad,null,204
107,Grace,31,Finance,null,Pune,2022-12-01,203
108,Henry,26,IT,60000,Bangalore,2024-01-10,null
